In [41]:
import urllib.request
import PIL
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

In [42]:
## Open the image
imgURL = "https://raw.githubusercontent.com/larvalabs/cryptopunks/master/punks.png"
urllib.request.urlretrieve(imgURL, "cryptoPunksAll.jpg")
img = PIL.Image.open("cryptoPunksAll.jpg").convert("RGB")
imgArray = np.asarray(img)

n = 10000

finalArray = np.empty((n, 24, 24, 3))
for i in range(100):
  for j in range(100):
    a, b = 24 * i, 24 * (i + 1)
    c, d = 24 * j, 24 * (j + 1)
    idx = j + i * (100)
    finalArray[idx,:,:,:] = imgArray[a:b,c:d,:]

In [43]:
temp  =  finalArray[0,:,:,:].copy()

d2min, d2max = 9,14
d1min, d1max = 11,17

temp[d1min : (d1max + 1), d2min : (d2max + 1)] = 255

# plt.imshow(temp.astype('uint8'))

In [44]:
cancerpunks = finalArray.copy()
label = np.zeros(n)

## Loop over the cryptopunks
for i in range(10000):
  flip = np.random.randint(0, 2)
  if flip is 1:
    label[i] = 1
    d1loc = np.random.randint(d1min, d1max + 1)
    d2loc = np.random.randint(d2min, d2max + 1)
    cancerpunks[i,d1loc,d2loc,:] = 255


<>:7: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
<>:7: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
C:\Users\49498\AppData\Local\Temp\ipykernel_6016\2969927026.py:7: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
  if flip is 1:


In [45]:
# ## plot some examples
# plt.figure(figsize=(10,10))
# for i in range(25):
#   plt.subplot(5,5,i+1)
#   plt.xticks([])
#   plt.yticks([])
#   plt.imshow(cancerpunks[i,:,:,:].astype('uint8'))
#   plt.title(label[i])

In [46]:
# Data preparation
x = np.transpose(cancerpunks/255, (0, 3, 1, 2))
x_train, x_test, y_train, y_test = train_test_split(x, label, test_size=0.15, random_state=42)

x_train_t = torch.tensor(x_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [ ]:
# Model definition
model = nn.Sequential(
    nn.Conv2d(3, 16, 3), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(16*11*11, 32), nn.ReLU(),
    nn.Linear(32, 1), nn.Sigmoid()
)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()
batch_size = 32

for epoch in range(50):
    permutation = torch.randperm(x_train_t.size()[0])
    for i in range(0, x_train_t.size()[0], batch_size):
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = x_train_t[indices], y_train_t[indices]
        optimizer.zero_grad()
        loss = criterion(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        if (epoch+1) % 5 == 0 and i == 0:
            print(f"Epoch {epoch+1} completed. Loss: {loss.item():.4f}")

Epoch 5 completed. Loss: 0.3868
Epoch 10 completed. Loss: 0.2397
Epoch 15 completed. Loss: 0.0950
Epoch 20 completed. Loss: 0.0506
Epoch 25 completed. Loss: 0.0417
Epoch 30 completed. Loss: 0.1167
Epoch 35 completed. Loss: 0.1370
Epoch 40 completed. Loss: 0.0330
Epoch 45 completed. Loss: 0.0126
Epoch 50 completed. Loss: 0.0134


In [48]:
# Evaluation
model.eval()
train_preds = (model(x_train_t) > 0.5).int().numpy().flatten()
test_preds = (model(x_test_t) > 0.5).int().numpy().flatten()

In [49]:
# Report metrics on train set
tn, fp, fn, tp = confusion_matrix(y_train, train_preds).ravel()
print(f'Train')
print(f'Accuracy: {accuracy_score(y_train, train_preds):.4f}')
print(f'Sensitivity: {tp/(tp+fn):.4f}')
print(f'Specificity: {tn/(tn+fp):.4f}')

Train
Accuracy: 0.9801
Sensitivity: 0.9610
Specificity: 0.9993


In [50]:
# Report metrics on test set
tn, fp, fn, tp = confusion_matrix(y_test, test_preds).ravel()
print(f'Test')
print(f'Accuracy: {accuracy_score(y_test, test_preds):.4f}')
print(f'Sensitivity: {tp/(tp+fn):.4f}')
print(f'Specificity: {tn/(tn+fp):.4f}')

Test
Accuracy: 0.9567
Sensitivity: 0.9259
Specificity: 0.9879
